In [1]:
# Construye el modelo
import pyomo.environ as pe

# Resuelve el modelo
import pyomo.opt as po

In [2]:
model = pe.ConcreteModel()

### Sets


In [3]:
refineries = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
pipelines = [1, 2]
connections = [
    ('A', 'B'), ('B', 'C'), ('C', 'D'),
    ('D', 'E'), ('E', 'F'), ('F', 'A'),
    ('A', 'G'), ('B', 'G'), ('C', 'G'),
    ('D', 'G'), ('E', 'G'), ('F', 'G')
]

model.refineries = pe.Set(initialize=refineries)
model.pipelines = pe.Set(initialize=pipelines)
model.connections = pe.Set(dimen=2, initialize=connections)

### Parameters


In [4]:
capacity_dict = {
    ('A', 'B'): 600, ('B', 'C'): 700, ('C', 'D'): 700,
    ('D', 'E'): 650, ('E', 'F'): 550, ('F', 'A'): 700,
    ('A', 'G'): 850, ('B', 'G'): 950, ('C', 'G'): 600,
    ('D', 'G'): 550, ('E', 'G'): 700, ('F', 'G'): 650
}
model.capacity = pe.Param(model.connections, initialize=capacity_dict)

In [5]:
demand_dict = {'A': 700, 'B': 650, 'C': 450, 'D': 570, 'E': 490, 'F': 630, 'G': 810}
penalty_dict = {'A': 300, 'B': 300, 'C': 400, 'D': 300, 'E': 450, 'F': 340, 'G': 350}
new_capacity_dict = {1: 1500, 2: 2500}

model.demand = pe.Param(model.refineries, initialize=demand_dict)
model.penalty = pe.Param(model.refineries, initialize=penalty_dict)
model.new_capacity = pe.Param(model.pipelines, initialize=new_capacity_dict)

### Variables


In [6]:
# x[i, p] = 1 si la tubería p se conecta a la refinería i
model.x = pe.Var(model.refineries, model.pipelines, domain=pe.Binary)
model.supply = pe.Var(model.refineries, domain=pe.NonNegativeReals)

def flow_bounds(model, i, j):
    return (-model.capacity[i, j], model.capacity[i, j])

model.flow = pe.Var(model.connections, domain=pe.Reals, bounds=flow_bounds)

def shortage_bounds(model, i):
    return (0, model.demand[i])

model.shortage = pe.Var(model.refineries, domain=pe.NonNegativeReals, bounds=shortage_bounds)

### Funcion objetivo


Minimizar la penalización por la demanda no atendida.


In [7]:
def obj_rule(model):
    return sum(model.penalty[i] * model.shortage[i] for i in model.refineries)

model.obj = pe.Objective(rule=obj_rule, sense=pe.minimize)

### Restricciones


In [8]:
def choose_refinery(model, p):
    return sum(model.x[i, p] for i in model.refineries) == 1

model.choose_refinery = pe.Constraint(model.pipelines, rule=choose_refinery)

def non_adjacent(model, i, j):
    return sum(model.x[i, p] + model.x[j, p] for p in model.pipelines) <= 1

model.non_adjacent = pe.Constraint(model.connections, rule=non_adjacent)

In [9]:
def supply_capacity(model, i):
    return model.supply[i] <= sum(model.new_capacity[p] * model.x[i, p] for p in model.pipelines)

model.supply_capacity = pe.Constraint(model.refineries, rule=supply_capacity)

def balance(model, i):
    inflow = sum(model.flow[a, b] for a, b in model.connections if b == i)
    outflow = sum(model.flow[a, b] for a, b in model.connections if a == i)
    return model.supply[i] + inflow - outflow + model.shortage[i] == model.demand[i]

model.balance = pe.Constraint(model.refineries, rule=balance)

## Resolución con gurobi


In [10]:
solver = po.SolverFactory("gurobi_direct")

results = solver.solve(model, tee=False)

In [11]:
for p in model.pipelines:
    for i in model.refineries:
        if pe.value(model.x[i, p]) > 0.5:
            print(f"Tubería {p} -> {i}: {pe.value(model.supply[i]):.2f} l/s")

print(f"PENALIZACIÓN MÍNIMA: {pe.value(model.obj):.2f} €")
print(f"Déficit total: {sum(pe.value(model.shortage[i]) for i in model.refineries):.2f} l/s")

print("\nDemanda atendida y déficit (l/s):")
for i in model.refineries:
    shortage = pe.value(model.shortage[i])
    served = pe.value(model.demand[i]) - shortage
    print(f"{i}: atendida {served:.2f}; déficit {shortage:.2f}")

print("\nFlujos (l/s):")
for i, j in model.connections:
    flow = pe.value(model.flow[i, j])
    if abs(flow) > 1e-6:
        a, b = (i, j) if flow > 0 else (j, i)
        print(f"{a} -> {b}: {abs(flow):.2f}")

Tubería 1 -> E: 1500.00 l/s
Tubería 2 -> A: 2500.00 l/s
PENALIZACIÓN MÍNIMA: 90000.00 €
Déficit total: 300.00 l/s

Demanda atendida y déficit (l/s):
A: atendida 700.00; déficit 0.00
B: atendida 650.00; déficit 0.00
C: atendida 450.00; déficit 0.00
D: atendida 270.00; déficit 300.00
E: atendida 490.00; déficit 0.00
F: atendida 630.00; déficit 0.00
G: atendida 810.00; déficit 0.00

Flujos (l/s):
A -> B: 250.00
B -> C: 550.00
C -> D: 170.00
E -> D: 650.00
F -> E: 340.00
A -> F: 700.00
A -> G: 850.00
G -> B: 950.00
G -> C: 70.00
D -> G: 550.00
E -> G: 700.00
G -> F: 270.00
